In [ ]:
import sys
import os
import torch
import transformers

# 1. Liberar el espacio de nombres ocupado por la versión nativa de Transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

# 2. Inyección de la ruta del repositorio local
repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 3. Importación directa desde la estructura modular de la comunidad
from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

model_path = "../weights/vibevoice-1.5b"

# 4. Diagnóstico de carga en la GPU
torch.cuda.empty_cache()
vram_inicial = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM Libre Inicial: {vram_inicial:.2f} GB")

try:
    print("Cargando VibeVoice 1.5B en hardware local...")
    model = VibeVoiceForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    vram_final = torch.cuda.memory_allocated() / 1024**3
    print(f"\n Conexión exitosa. El entorno local está operativo.")
    print(f"VRAM Consumida por el Modelo: {vram_final - vram_inicial:.2f} GB")
    print(f"VRAM Total Ocupada en GPU: {vram_final:.2f} GB / 8.00 GB")

except Exception as e:
    print(f"\n Error al mapear los tensores en la GPU: {e}")